# 06｜目标检测中的 TP、FP、FN、Precision 与 Recall

上一课完成了推理后处理：模型先产生许多预测，再经过分数筛选和 NMS 得到最终候选结果。

现在需要评价这些结果究竟好不好。目标检测不能只看类别是否正确，因为一个“狗”预测还必须真正框住某只狗，而且同一个真实目标不能被重复计分。

本课只学习 TP、FP、FN、Precision 和 Recall。PR 曲线、AP 与 mAP 留到下一课。

## 1. 为什么分类准确率不能直接评价目标检测

图像分类通常是一张图片对应一个类别，可以直接统计预测类别是否正确。

目标检测的一张图片却可能有多个真实物体和多个预测框。即使预测类别写着“狗”，仍然可能出现：

- 框的位置完全没有覆盖真实狗。
- 框只覆盖了狗的一小部分。
- 两个框重复预测同一只狗。
- 图片中另一只狗完全漏检。

因此，判断一条检测是否正确，必须同时检查类别和边界框重叠。

## 2. 评价时先确定类别和 IoU 标准

为了判断某条预测能否命中真实目标，通常至少要求：

1. 预测类别与真实类别相同。
2. 预测框与真实框的 IoU 达到规定阈值。

例如当前规定：

$$
t_{eval}=0.5
$$

那么一条“狗”预测只有与某个尚未被匹配的真实狗框满足：

$$
\operatorname{IoU}\geq0.5
$$

才有机会被计为一次正确检测。阈值 0.5 只是本课示例，实际评价协议可能使用一个或多个 IoU 阈值。

## 3. 评价通常按类别分别进行

评价“狗”类别时，主要比较：

- 所有预测为狗的框。
- 数据集中所有真实狗框。

猫、汽车等其他类别会分别统计。

这样才能回答：模型检测狗的能力如何、检测猫的能力如何，而不是把不同类别的框混在一起匹配。

本课的完整例子只评价狗类别。

## 4. 什么是 TP

TP 是 True Positive，可以理解为正确检出的目标。

一条预测成为 TP，通常要同时满足：

- 类别正确。
- 与某个尚未匹配的同类真实框达到 IoU 阈值。

例如预测类别为狗，与一只真实狗的 IoU 为 0.82，而且这只真实狗尚未被其他更高分预测匹配，那么这条预测记为一个 TP。

TP 统计的是成功检测次数。

## 5. 什么是 FP

FP 是 False Positive，可以理解为模型给出了一条检测结果，但它不能对应一个新的正确目标。

常见 FP 包括：

- 把背景误报成物体。
- 类别预测错误。
- 框与同类真实目标的 IoU 不达标。
- 重复检测一个已经被更高分预测匹配的真实物体。

FP 统计的是误报或无效预测次数。

## 6. 什么是 FN

FN 是 False Negative，可以理解为真实存在，但模型没有成功检出的目标。

所有预测处理完以后，如果某个真实物体仍未被任何合格预测匹配，它就是一个 FN。

例如图片里有三只真实的狗，最终只有两只狗被正确预测，那么：

$$
FN=1
$$

FN 统计的是漏检次数。

## 7. 为什么每个真实目标最多匹配一次

假设同一只狗附近有三个预测框，而且三个框的 IoU 都超过阈值。如果三条预测全部记为 TP，模型只检测到一只狗，却会得到三次正确计数，这显然不合理。

因此，评价时每个真实目标最多贡献一个 TP：

- 分数最高且满足条件的预测先匹配真实目标，记为 TP。
- 后续预测即使与同一真实目标高度重叠，因为该目标已经被占用，也只能记为 FP。

这就是为什么评价前要把预测按检测分数从高到低排序。

## 8. 建立一个完整例子

假设整个评价集合中有三只真实的狗：

$$
G_1,G_2,G_3
$$

模型给出五条狗类别预测，并已按检测分数从高到低排列：

| 预测 | 分数 | 最接近的真实狗 | IoU | 直观情况 |
|---:|---:|---:|---:|---|
| $P_1$ | 0.95 | $G_1$ | 0.82 | 很好地框住第一只狗 |
| $P_2$ | 0.90 | $G_2$ | 0.73 | 很好地框住第二只狗 |
| $P_3$ | 0.70 | $G_2$ | 0.64 | 重复框住第二只狗 |
| $P_4$ | 0.60 | $G_3$ | 0.28 | 没有准确框住第三只狗 |
| $P_5$ | 0.45 | $G_3$ | 0.75 | 较低分但准确框住第三只狗 |

评价 IoU 阈值仍设为 0.5。

## 9. 逐条判断 P1 和 P2

### 预测 $P_1$

$P_1$ 与尚未匹配的 $G_1$ 的 IoU 为 0.82，超过 0.5，因此：

$$
P_1=TP,\qquad G_1\text{ 标记为已匹配}
$$

### 预测 $P_2$

$P_2$ 与尚未匹配的 $G_2$ 的 IoU 为 0.73，也超过 0.5，因此：

$$
P_2=TP,\qquad G_2\text{ 标记为已匹配}
$$

## 10. 重复预测 P3 为什么是 FP

$P_3$ 与 $G_2$ 的 IoU 为 0.64，看起来超过了阈值。

但是 $G_2$ 已经被分数更高的 $P_2$ 成功匹配。一个真实目标不能再次贡献 TP，所以：

$$
P_3=FP
$$

这条预测的定位可能并不差，但它没有发现一个新的真实物体，只是重复报告了已经检测到的第二只狗。

因此，NMS 删除重复框不仅让展示结果更整洁，也能减少评价中的重复 FP。

## 11. P4 和 P5 怎样判断

### 预测 $P_4$

$P_4$ 与尚未匹配的 $G_3$ 的 IoU 只有 0.28，没有达到 0.5，因此：

$$
P_4=FP
$$

### 预测 $P_5$

$P_5$ 与 $G_3$ 的 IoU 为 0.75。虽然分数只有 0.45，但如果当前分数阈值允许它进入评价，它可以成功匹配 $G_3$：

$$
P_5=TP,\qquad G_3\text{ 标记为已匹配}
$$

这说明检测分数和边界框质量相关，但不是同一个数。较低分预测也可能拥有准确边界框。

## 12. Precision 衡量什么

Precision 中文常译为精确率。它关心：模型输出的这些检测中，有多少是真的正确检测。

$$
\operatorname{Precision}=\frac{TP}{TP+FP}
$$

如果模型输出 10 条检测，其中 8 条正确、2 条误报：

$$
\operatorname{Precision}=\frac{8}{8+2}=0.8
$$

Precision 高，说明模型给出的结果比较可靠、误报较少。

## 13. Recall 衡量什么

Recall 中文常译为召回率。它关心：所有真实物体中，有多少被模型成功找到。

$$
\operatorname{Recall}=\frac{TP}{TP+FN}
$$

如果数据中有 10 个真实目标，模型成功检测出 7 个、漏掉 3 个：

$$
\operatorname{Recall}=\frac{7}{7+3}=0.7
$$

Recall 高，说明模型漏掉的真实物体较少。

## 14. 在完整例子中计算 Precision 和 Recall

如果分数阈值不高于 0.45，五条预测都进入评价，最终：

$$
TP=3,\qquad FP=2,\qquad FN=0
$$

因此：

$$
\operatorname{Precision}=\frac{3}{3+2}=0.6
$$

$$
\operatorname{Recall}=\frac{3}{3+0}=1.0
$$

模型找全了三只狗，所以 Recall 为 1；但五条检测里有两条 FP，所以 Precision 只有 0.6。

## 15. 提高分数阈值会发生什么

如果把分数阈值提高到 0.60，只保留 $P_1$、$P_2$、$P_3$、$P_4$，较低分但能命中 $G_3$ 的 $P_5$ 被提前删除。

此时：

$$
TP=2,\qquad FP=2,\qquad FN=1
$$

$$
\operatorname{Precision}=\frac{2}{4}=0.5,\qquad
\operatorname{Recall}=\frac{2}{3}\approx0.667
$$

在这个具体例子中，提高阈值反而同时降低了 Precision 和 Recall，因为被删掉的 $P_5$ 是有效预测，而 $P_3$、$P_4$ 两条 FP 仍被保留。

这提醒我们：阈值变化对指标的具体影响取决于预测分数排序，不能只凭一句口号判断。

## 16. 一般情况下的 Precision–Recall 权衡

虽然单个例子可能有不同变化，但在整个数据集上逐渐降低分数阈值时，通常会：

- 接纳更多预测。
- 找到更多真实目标，Recall 往往提高或保持不变。
- 同时引入更多误报，Precision 可能下降。

逐渐提高分数阈值时则通常相反：结果更谨慎，误报可能减少，但也更容易漏掉低分真实目标。

下一课会系统改变分数阈值，得到一系列 Precision 与 Recall 点，并连接成 PR 曲线。

## 17. 为什么目标检测中很少强调 TN

TN 是 True Negative，即正确判断为背景。

分类任务通常有明确数量的负样本，因此容易统计 TN。但图像中的背景位置几乎可以无限细分：

- 每个没有物体的像素可以算背景。
- 每个可能的矩形区域也可以算背景。
- 不同检测器产生的候选数量还不同。

因此，目标检测评价通常围绕 TP、FP 和 FN 构建 Precision 与 Recall，而不把庞大且定义依赖检测结构的 TN 作为核心。

## 18. 类别预测错误时怎样计数

假设一只真实的狗被模型框得很准确，但预测类别是猫。

按类别分别评价时：

- 对狗类别来说，这只真实狗没有被狗预测匹配，因此产生一个 FN。
- 对猫类别来说，这条猫预测没有对应的真实猫，因此产生一个 FP。

所以边界框准确并不足以成为 TP，类别必须同时正确。

## 19. 指标通常在整个数据集上累计

单张图片的例子有助于理解，但正式评价通常会对某个类别收集整个验证集或测试集的预测。

然后：

$$
\text{整个数据集的该类别预测}
\rightarrow\text{按分数排序}
\rightarrow\text{逐条判定 TP 或 FP}
\rightarrow\text{累计 Recall 与 Precision}
$$

这样指标反映的是模型在大量不同图片上的整体检测能力，而不是某一张图片的偶然表现。

## 20. 评价匹配、目标分配和 NMS 的区别

| 步骤 | 所在阶段 | 比较对象 | 主要目的 |
|---|---|---|---|
| 训练目标分配 | 训练 | 候选或预测与真实目标 | 产生训练监督 |
| NMS | 推理后处理 | 预测框与预测框 | 删除重复结果 |
| 评价匹配 | 模型评价 | 预测与真实目标 | 统计 TP、FP、FN |

三者都可能使用 IoU，所以容易混淆，但它们的阶段和目的不同。

评价时的“一份真实目标最多匹配一次”只是为了公平计数，不等于 DETR 训练时的匈牙利一对一匹配。

## 21. 常见误区

1. 预测类别正确但框不达 IoU 标准，不能记为 TP。
2. 与已匹配真实目标重复的预测通常记为 FP，而不是第二个 TP。
3. FN 对应漏掉的真实目标，不是某一条错误预测。
4. Precision 的分母是模型输出的有效预测数量，Recall 的分母是真实目标数量。
5. NMS 的 IoU 阈值与评价 TP 的 IoU 阈值不是同一个用途。
6. 正式指标通常按类别并在整个数据集上统计。

## 22. 本节小结

这一课需要真正记住七个结论：

1. 检测 TP 必须同时满足类别正确和边界框 IoU 达标。
2. 每个真实目标最多匹配一条预测，后续重复预测会成为 FP。
3. FP 表示误报、类别错误、定位不达标或重复预测。
4. FN 表示真实存在但没有成功检测出的物体。
5. Precision 关心输出结果中有多少正确，Recall 关心真实目标中有多少被找到。
6. 分数阈值变化会改变保留的预测，从而改变 Precision 和 Recall。
7. 训练目标分配、NMS 和评价匹配是三个不同步骤。

$$
\operatorname{Precision}=\frac{TP}{TP+FP},\qquad
\operatorname{Recall}=\frac{TP}{TP+FN}
$$

下一课将把不同分数阈值下的 Precision 与 Recall 连成 PR 曲线，再理解 AP 为什么是曲线下的综合面积，以及 mAP 为什么要对类别和 IoU 标准做汇总。

## 23. 自测问题

1. 一条预测成为 TP 通常需要满足哪两个条件？
2. 为什么一个真实目标最多匹配一条预测？
3. 重复检测同一真实目标的后续预测通常记为什么？
4. FP 常见的四种来源是什么？
5. FN 对应预测还是未被检测出的真实目标？
6. Precision 与 Recall 的分母分别是什么？
7. 在完整例子中，为什么 $P_3$ 的 IoU 达标却仍然是 FP？
8. 为什么 $P_5$ 分数较低却可以成为 TP？
9. 类别错误但定位准确的预测会怎样影响两个类别？
10. 为什么目标检测评价通常不强调 TN？
11. 目标分配、NMS 和评价匹配分别解决什么问题？
12. 为什么不能只在一张图片上判断整个检测器好坏？

### 自测参考答案

1. 预测类别正确，并且与尚未匹配的同类真实框达到规定 IoU 阈值。
2. 否则多个重复框可以让同一个真实物体贡献多次正确计数。
3. FP。
4. 背景误报、类别错误、定位 IoU 不达标，以及重复检测已匹配目标。
5. 未被成功检测的真实目标。
6. Precision 的分母是 $TP+FP$，Recall 的分母是 $TP+FN$。
7. 它最接近的 $G_2$ 已经被分数更高的 $P_2$ 匹配。
8. 检测分数与边界框 IoU不是同一个量；只要进入评价且满足匹配条件，它仍可成为 TP。
9. 对真实类别造成一个 FN，对错误预测类别造成一个 FP。
10. 图像背景区域几乎可以无限划分，TN 数量缺少统一且有意义的定义。
11. 目标分配产生训练监督；NMS 删除预测之间的重复；评价匹配统计预测与真实目标的 TP、FP、FN。
12. 单图结果具有偶然性，正式指标应反映整个数据集上的总体表现。